In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import proplot as pplt
import matplotlib as mpl
import matplotlib.dates as dates
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#dir = 'D:/yuanqi/new_pdm/seperatebaseflow/'
dir_topo = 'D:/yuanqi/new urania partition/camels_600.txt'
camels_info = pd.read_csv(dir_topo, delimiter=';')
camels_info = camels_info.sort_values(by='aridity').reset_index(drop=True)
gauge_id = camels_info['gauge_id'].values.astype(np.int64)

In [ ]:
def timeseries_coverter(data_array, start_yr, ending_yr):
    #import numpy as np
    #from datetime import date, timedelta
    sdate = date(start_yr,1,1)
    edate = date(ending_yr, 12, 31)
    data_ts = pd.DataFrame(data_array)

    data_ts.index = pd.date_range(start=sdate, end=edate, freq='D')

    mean_monthly = np.squeeze(np.array(data_ts.resample('M').sum()))

    return mean_monthly

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import proplot as pplt

# -----------------------------
# Parameters
# -----------------------------
MONTHS = ['J','F','M','A','M','J','J','A','S','O','N','D']
MONTH_TO_SEASON = {
    0:'winter', 1:'winter', 11:'winter',
    2:'spring',3:'spring',4:'spring',
    5:'summer',6:'summer',7:'summer',
    8:'autumn',9:'autumn',10:'autumn'
}
THRESH = 0.4       # fraction threshold for seasonal dominant

# -----------------------------
# Main loop
# -----------------------------
results = []
regimes_dict = {}

for stationid in gauge_id:
    sid_float = float(stationid)
    camels_info['gauge_id'] = camels_info['gauge_id'].astype(float)
    aridity = camels_info[camels_info['gauge_id']==sid_float]['aridity'].iloc[0]
    aridity = round(aridity,3)
    stationid_str = "%08d"%int(sid_float)

    # -----------------------------
    # read simulated data
    # -----------------------------
    outd4r = pd.read_csv(f'D:/yuanqi/1106/sim/{stationid_str}_AI_{aridity}_simulated26.csv', index_col=0)
    outd4 = outd4r[730:]  # skip spinup
    ri_ts = outd4['ri']
    rs_all = outd4['rs']
    rg = outd4['rg']
    qb_ts = outd4['qb']
    rs_ts = np.array(rs_all - rg)
    dqsim4 = outd4['Qsim']

    # -----------------------------
    # monthly mean
    # -----------------------------
    ri_mon = timeseries_coverter(ri_ts,1987,2014)
    rs_mon = timeseries_coverter(rs_ts,1987,2014)
    qb_mon = timeseries_coverter(qb_ts,1987,2014)
    sim_mon = timeseries_coverter(dqsim4,1987,2014)

    ri_monthly = np.reshape(ri_mon,(-1,12)).mean(axis=0)
    rs_monthly = np.reshape(rs_mon,(-1,12)).mean(axis=0)
    qb_monthly = np.reshape(qb_mon,(-1,12)).mean(axis=0)
    sim_monthly = np.reshape(sim_mon,(-1,12)).mean(axis=0)

    # -----------------------------
    # Determine seasonal dominant components
    # -----------------------------
    peaks = {'ri': np.argmax(ri_monthly),
             'rs': np.argmax(rs_monthly),
             'qb': np.argmax(qb_monthly)}

    seasonal_tags = []
    for comp, peak_idx in peaks.items():
        total = ri_monthly[peak_idx]+rs_monthly[peak_idx]+qb_monthly[peak_idx]+1e-12
        frac = {'ri': ri_monthly[peak_idx]/total,
                'rs': rs_monthly[peak_idx]/total,
                'qb': qb_monthly[peak_idx]/total}[comp]

        # Only keep components exceeding threshold
        if frac >= THRESH:
            season_label = MONTH_TO_SEASON[peak_idx]
            tag = f"{comp}_{season_label}_domi"
            if tag not in seasonal_tags:
                seasonal_tags.append(tag)

    if seasonal_tags:
        final_tag_str = "+".join(seasonal_tags)
    else:
        final_tag_str = "others"

    results.append((stationid_str, final_tag_str))

    # -----------------------------
    # Save monthly regimes for plotting
    # -----------------------------
    regimes_dict[stationid_str] = pd.DataFrame({
        "mon": MONTHS,
        "sim": sim_monthly,
        "ri": ri_monthly,
        "rs": rs_monthly,
        "qb": qb_monthly
    })

# -----------------------------
# Save CSV
# -----------------------------
df_results = pd.DataFrame(results, columns=["station_id","seasonal_domi"])
csv_path = "D:/yuanqi/new urania partition/seasonal_domi_combined.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df_results.to_csv(csv_path,index=False)


In [ ]:
import pandas as pd

# Load original results
groups = pd.read_csv("D:/yuanqi/new urania partition/seasonal_domi_combined.csv")

# Count each group
group_counts = groups['group_name'].value_counts()

# Identify small groups (<10)
small_groups = group_counts[group_counts < 10].index.tolist()

# Function to remap small groups
def remap_small_group_rs(name):
    """
    Remap small groups into rs_cold, rs_warm, or no_rs
    """
    tokens = name.split("+")
    has_rs_cold = False
    has_rs_warm = False
    for token in tokens:
        parts = token.split("_")
        if len(parts) < 2:
            continue
        comp = parts[0]  # component: ri, rs, qb
        season = parts[1]  # season: spring, summer, autumn, winter
        if comp != "rs":
            continue
        if season in ["spring", "winter"]:
            has_rs_cold = True
        elif season in ["summer", "autumn"]:
            has_rs_warm = True
    # Determine new group
    if has_rs_cold:
        return "rs_cold"
    elif has_rs_warm:
        return "rs_warm"
    else:
        return "no_rs"

# Copy original groups
groups_new = groups.copy()

# Only remap small groups
groups_new.loc[groups_new['group_name'].isin(small_groups), 'merged_group'] = \
    groups_new.loc[groups_new['group_name'].isin(small_groups), 'group_name'].apply(remap_small_group_rs)

# For large groups, keep original
groups_new.loc[~groups_new['group_name'].isin(small_groups), 'merged_group'] = \
    groups_new.loc[~groups_new['group_name'].isin(small_groups), 'group_name']

# Check results
print(groups_new['merged_group'].value_counts())

# Optionally save
groups_new.to_csv("D:/yuanqi/new urania partition/seasonal_domi_regroup_rs.csv", index=False)


merged_group
qb_spring_domi                   134
rs_spring_domi                    80
rs_spring_domi+qb_spring_domi     77
rs_winter_domi                    51
ri_spring_domi                    45
no_rs                             41
rs_cold                           29
rs_spring_domi+qb_summer_domi     25
qb_summer_domi                    21
ri_summer_domi                    17
rs_warm                           17
ri_spring_domi+rs_spring_domi     16
rs_summer_domi                    14
rs_winter_domi+qb_spring_domi     12
ri_spring_domi+qb_summer_domi     11
others                            10
Name: count, dtype: int64
